In [0]:
%pip install aiohttp pandas
dbutils.library.restartPython()

In [0]:
TWITCH_CLIENT_ID = "knstk5irrjnfrvjcg2p3f64od4wjae"
TWITCH_CLIENT_SECRET = "i6ubcr4vchw4uqwaro0132uftpntat"

In [0]:
import aiohttp
import asyncio

# Solicita un access token OAuth 2.0 a Twitch usando Client Credentials Flow.
async def get_twitch_token(client_id, client_secret):
    # Endpoint oficial de Twitch para emisión de tokens.
    url = "https://id.twitch.tv/oauth2/token"
    
    # Parámetros requeridos por Twitch para autenticar la aplicación.
    params = {
        "client_id": client_id,
        "client_secret": client_secret,
        "grant_type": "client_credentials"
    }

    # Abre una sesión HTTP asíncrona.
    async with aiohttp.ClientSession() as session:
        # Envía una petición POST al endpoint del token.
        async with session.post(url, params=params) as response:
            # Lanza un error si la respuesta HTTP no fue exitosa.
            response.raise_for_status()
            
            # Devuelve la respuesta en formato JSON.
            return await response.json()

In [0]:
import nest_asyncio
nest_asyncio.apply()
import asyncio

token_data = asyncio.get_event_loop().run_until_complete(get_twitch_token(TWITCH_CLIENT_ID, TWITCH_CLIENT_SECRET))
print(token_data)
access_token = token_data["access_token"]

In [0]:
import aiohttp

async def validate_twitch_token(access_token):
    url = "https://id.twitch.tv/oauth2/validate"
    headers = {
        "Authorization": f"OAuth {access_token}"
    }

    async with aiohttp.ClientSession() as session:
        async with session.get(url, headers=headers) as response:
            response.raise_for_status()
            return await response.json()
        
token_info = await validate_twitch_token('dtmpl68w86mtt9t8ztgj0a33cw0efg')
print(token_info)

In [0]:
async def search_twitch_categories(query, client_id, access_token):
    url = "https://api.twitch.tv/helix/search/categories"
    
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Client-Id": client_id
    }
    
    params = {
        "query": query
    }

    async with aiohttp.ClientSession() as session:
        async with session.get(url, headers=headers, params=params) as response:
            response.raise_for_status()
            return await response.json()

In [0]:
twitch_response = await search_twitch_categories(
    "music",
    TWITCH_CLIENT_ID,
    access_token
)

print(twitch_response)

In [0]:
import pandas as pd

categories = twitch_response.get("data", [])

twitch_df = (
    pd.json_normalize(categories)
    
)

twitch_df.head()

In [0]:
spark_df = spark.createDataFrame(twitch_df)

spark_df.write.mode("overwrite").option("mergeSchema", "true").format("delta").saveAsTable("bronze.twitch_categories")

In [0]:
display(spark.table("bronze.twitch_categories"))
print(spark.table("bronze.twitch_categories").count())